# Who should get the e-mail? Analysis notebook

This notebook walks through the analysis behind the [live dashboard](https://mehtaaarush.github.io/ab-testing-uplift-modeling/).
All logic lives in the `uplift` package, so the numbers here match the pipeline exactly.

**The experiment.** 64,000 customers who bought from an online retailer in the past year were randomly split three ways: a men's merchandise
e-mail, a women's merchandise e-mail, or no e-mail. Visits, conversions and spend were tracked for the following two weeks.

**Outline**
1. Data and validation
2. Can the experiment be trusted? (sample ratio, covariate balance)
3. Average effects with confidence intervals and multiple-testing control
4. Is the revenue result robust? (bootstrap, regression adjustment)
5. Who responds? (segment effects)
6. Uplift models: can we predict who is persuadable?
7. Which targeting policy should the business run?
8. Power for the next test

In [1]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uplift import config, data, experiment, targeting
from uplift.export import ARM_NAMES
from uplift.figures import _style, ARM_COLORS, BASELINE, INK2

_style()
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Data and validation

`data.load()` downloads the file on first use and checks its SHA-256 hash, so a changed or truncated download fails loudly.
`tidy()` maps the arm labels to short keys, fixes a typo in the source (`Surburban`) and asserts the rules every analysis relies on:
no spend without a conversion, no conversion without a visit, binary outcomes and valid arms.

In [2]:
df = data.load()
print(f"{len(df):,} customers, {df.shape[1]} columns")
df.head()

64,000 customers, 15 columns


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,visit,conversion,spend,arm,purchase_mix,tenure,recency_band
0,10,$100–$200,142.4400,1,0,Suburban,0,Phone,0,0,0.0000,womens,Men's only,Returning customer,10–12 months
1,6,$200–$350,329.0800,1,1,Rural,1,Web,0,0,0.0000,control,Both,New customer,4–6 months
2,7,$100–$200,180.6500,0,1,Suburban,1,Web,0,0,0.0000,womens,Women's only,New customer,7–9 months
3,9,$500–$750,675.8300,1,0,Rural,1,Web,0,0,0.0000,mens,Men's only,New customer,7–9 months
4,2,$0–$100,45.3400,1,0,Urban,0,Web,0,0,0.0000,womens,Men's only,Returning customer,1–3 months


In [3]:
summary = df.groupby("arm").agg(customers=("visit", "size"), visit_rate=("visit", "mean"),
                                conversion_rate=("conversion", "mean"), revenue_per_customer=("spend", "mean"))
summary.index = summary.index.map(ARM_NAMES)
summary

,customers,visit_rate,conversion_rate,revenue_per_customer
arm,,,,
No e-mail,21306,0.1062,0.0057,0.6528
Men's e-mail,21307,0.1828,0.0125,1.4226
Women's e-mail,21387,0.1514,0.0088,1.0772


## 2. Can the experiment be trusted?

Two checks before reading any result. **Sample-ratio mismatch**: are the arms the planned (equal) size? A broken assignment or logging bug
usually shows up here first. **Covariate balance**: do the arms look alike before treatment? The standardised mean difference (SMD) should
be well under 0.1 for every pre-treatment covariate.

In [4]:
srm = experiment.srm_test(df)
print(f"Arm sizes: {srm['counts']}")
print(f"Chi-square = {srm['chi2']:.3f}, p = {srm['p_value']:.3f} -> {'passed' if srm['passed'] else 'FAILED'}")
balance = experiment.balance_table(df)
balance.columns = [ARM_NAMES[c] for c in balance.columns]
balance.round(4)

Arm sizes: {'control': 21306, 'mens': 21307, 'womens': 21387}
Chi-square = 0.203, p = 0.904 -> passed


,Men's e-mail,Women's e-mail
Recency (months),0.0068,0.0052
Past-year spend ($),0.0076,0.0065
Bought men's items,-0.0046,-0.0086
Bought women's items,0.0076,0.0049
New customer,-0.0009,0.0026
Area: urban,0.0020,-0.0018
Area: suburban,-0.0117,-0.0011
Area: rural,0.0137,0.0040
Channel: web,0.0110,-0.0051
Channel: phone,-0.0083,0.0086


Both checks pass: the split is what chance alone would produce (p = 0.90), and the largest absolute SMD is 0.014. Differences between the arms
can be attributed to the e-mails.

## 3. Average effects

Rates use a two-proportion z-test (pooled for the p-value, unpooled Wald interval for the difference), revenue uses Welch's t-test, and
relative lifts get delta-method intervals on the log ratio. The six p-values are Holm-adjusted as one family.

In [5]:
effects = experiment.ate_table(df)
show = effects.assign(arm=effects["arm"].map(ARM_NAMES), metric=effects["metric"].map(config.METRICS))
show[["arm", "metric", "control", "treatment", "diff", "ci_low", "ci_high", "lift", "p_value", "p_holm", "significant"]]

,arm,metric,control,treatment,diff,ci_low,ci_high,lift,p_value,p_holm,significant
0,Men's e-mail,Visit rate,0.1062,0.1828,0.0766,0.0700,0.0832,0.7214,0.0000,0.0000,True
1,Men's e-mail,Conversion rate,0.0057,0.0125,0.0068,0.0050,0.0086,1.1884,0.0000,0.0000,True
2,Men's e-mail,Revenue per customer,0.6528,1.4226,0.7698,0.4851,1.0545,1.1793,0.0000,0.0000,True
3,Women's e-mail,Visit rate,0.1062,0.1514,0.0452,0.0389,0.0516,0.4261,0.0000,0.0000,True
4,Women's e-mail,Conversion rate,0.0057,0.0088,0.0031,0.0015,0.0047,0.5433,0.0002,0.0003,True
5,Women's e-mail,Revenue per customer,0.6528,1.0772,0.4244,0.1690,0.6799,0.6502,0.0011,0.0011,True


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.6))
scales = {"visit": (100, "pp"), "conversion": (100, "pp"), "spend": (1, "$")}
for ax, metric in zip(axes, config.METRICS):
    k, unit = scales[metric]
    sub = effects[effects["metric"] == metric].reset_index(drop=True)
    for i, r in sub.iterrows():
        y = 1 - i
        ax.plot([r["ci_low"] * k, r["ci_high"] * k], [y, y], color=ARM_COLORS[r["arm"]], lw=2.2)
        ax.plot(r["diff"] * k, y, "o", color=ARM_COLORS[r["arm"]], ms=8)
    ax.axvline(0, color=BASELINE, lw=1)
    ax.set_yticks([1, 0], [ARM_NAMES[a] for a in sub["arm"]] if ax is axes[0] else ["", ""])
    ax.set_ylim(-0.6, 1.6)
    ax.set_title(f"{config.METRICS[metric]} ({unit})")
plt.show()

**Takeaways.** Both e-mails work on every outcome, and all six effects survive the Holm correction. The men's e-mail is stronger
everywhere: +7.7 pp in visits against +4.5 pp, and +$0.77 in revenue per customer against +$0.42. Relative lifts look dramatic
(conversion +119%), but absolute numbers are small: most of the response is browsing, not buying.

## 4. Is the revenue result robust?

Revenue is zero-inflated and heavy-tailed, which is exactly where a t-interval can mislead. Two cross-checks: a percentile bootstrap, and
Lin's (2013) regression adjustment, which uses pre-treatment covariates (with treatment interactions and HC2 errors) to soak up variance.

In [7]:
buyers = df.loc[df["spend"] > 0, "spend"]
print(f"Customers with any spend: {(df['spend'] > 0).mean():.2%}")
print(f"Spend among buyers: median ${buyers.median():.0f}, 95th percentile ${buyers.quantile(0.95):.0f}, max ${buyers.max():.0f}")
rows = []
for arm in config.TREATMENTS:
    welch = effects[(effects["arm"] == arm) & (effects["metric"] == "spend")].iloc[0]
    boot = experiment.bootstrap_mean_diff(df.loc[df["arm"] == arm, "spend"], df.loc[df["arm"] == "control", "spend"])
    lin = experiment.regression_adjusted_effect(df, arm, "spend")
    rows.append({"e-mail": ARM_NAMES[arm], "estimate": welch["diff"], "welch_ci": (round(welch["ci_low"], 3), round(welch["ci_high"], 3)),
                 "bootstrap_ci": tuple(round(v, 3) for v in boot), "lin_estimate": lin["diff"],
                 "lin_ci": (round(lin["ci_low"], 3), round(lin["ci_high"], 3)), "ci_narrower_by": lin["ci_width_reduction"]})
pd.DataFrame(rows)

Customers with any spend: 0.90%
Spend among buyers: median $81, 95th percentile $362, max $499


,e-mail,estimate,welch_ci,bootstrap_ci,lin_estimate,lin_ci,ci_narrower_by
0,Men's e-mail,0.7698,"(0.485, 1.055)","(0.486, 1.05)",0.7669,"(0.483, 1.051)",0.0013
1,Women's e-mail,0.4244,"(0.169, 0.68)","(0.175, 0.684)",0.4247,"(0.169, 0.68)",0.0009


The three intervals agree closely. Regression adjustment narrows them by about 0.1%: past spend, recency, channel and area barely predict
who buys within a two-week window. For visits it helps a little more (about 1.5%), but not enough to change any conclusion.

## 5. Who responds?

Each segment below is its own small A/B test. These are exploratory comparisons, so wide intervals deserve caution.

In [8]:
seg = experiment.segment_effects(df, "purchase_mix", "visit")
seg.assign(arm=seg["arm"].map(ARM_NAMES))[["level", "arm", "control", "treatment", "diff", "ci_low", "ci_high", "n_treatment"]]

,level,arm,control,treatment,diff,ci_low,ci_high,n_treatment
0,Men's only,Men's e-mail,0.0998,0.1690,0.0692,0.0596,0.0788,9558
1,Men's only,Women's e-mail,0.0998,0.1109,0.0111,0.0024,0.0197,9622
2,Both,Men's e-mail,0.1805,0.3150,0.1344,0.1091,0.1598,2181
3,Both,Women's e-mail,0.1805,0.2517,0.0711,0.0465,0.0957,2118
4,Women's only,Men's e-mail,0.0958,0.1664,0.0706,0.0611,0.0801,9568
5,Women's only,Women's e-mail,0.0958,0.1698,0.0740,0.0644,0.0835,9647


In [9]:
fig, ax = plt.subplots(figsize=(8, 3.2))
levels = ["Men's only", "Women's only", "Both"]
for arm, offset in [("mens", 0.14), ("womens", -0.14)]:
    rows = seg[seg["arm"] == arm].set_index("level").reindex(levels[::-1])
    y = np.arange(3) + offset
    ax.hlines(y, rows["ci_low"] * 100, rows["ci_high"] * 100, color=ARM_COLORS[arm], lw=2.2)
    ax.plot(rows["diff"] * 100, y, "o", color=ARM_COLORS[arm], ms=8, label=ARM_NAMES[arm])
ax.axvline(0, color=BASELINE, lw=1)
ax.set_yticks(range(3), levels[::-1])
ax.set_xlabel("Change in visit rate vs. no e-mail (pp)")
ax.set_title("Visit uplift by past purchases", pad=26)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
plt.show()

The women's e-mail does almost nothing for customers who only buy men's merchandise (+1.1 pp), while the men's e-mail lifts every group,
most of all customers who buy both (+13.4 pp). That heterogeneity is what an uplift model should be able to learn from the features.

## 6. Uplift models

Uplift models predict the *change* in visit probability caused by an e-mail, per customer. Each campaign (e-mail vs. no e-mail) gets its own model.
Visits are the target because conversions (about 1%) are too sparse to rank customers reliably.

* **T-learner**: separate outcome models for treated and control customers; uplift is the difference of their predictions.
* **S-learner**: one model with the treatment flag as a feature.
* **Class transformation** (Jaskowski & Jaroszewicz, 2012): with a 50/50 split, uplift = 2 P(z = 1 | x) − 1 for a transformed label z.

The data are split 70/30, stratified on arm and outcome. The model is chosen by five-fold cross-validated AUUC on the 70%; everything else
is measured on the untouched 30%. AUUC gain is the average height of the uplift curve above the random-targeting line, as a share of the total effect.

In [10]:
train, test = targeting.split(df)
results = {arm: targeting.evaluate_campaign(train, test, arm) for arm in config.TREATMENTS}
table = pd.concat({ARM_NAMES[arm]: pd.DataFrame(res["models"]).set_index("name") for arm, res in results.items()})
table[["cv_auuc", "test_auuc", "test_low", "test_high", "capture_30"]].round(3)

cv_auuc  test_auuc  \
               name                                                           
Men's e-mail   T-learner (gradient boosting)              0.0200     0.0020   
               S-learner (gradient boosting)              0.0020     0.0320   
               Class transformation (gradient boosting)  -0.0060    -0.0200   
               T-learner (logistic regression)            0.0220     0.0250   
Women's e-mail T-learner (gradient boosting)              0.1110     0.1370   
               S-learner (gradient boosting)              0.1360     0.1610   
               Class transformation (gradient boosting)   0.1090     0.1430   
               T-learner (logistic regression)            0.1800     0.1710   

                                                         test_low  test_high  \
               name                                                            
Men's e-mail   T-learner (gradient boosting)              -0.0490     0.0540   
               S-learner (gradient boosting)              -0.0160     0.0870   
               Class transformation (gradient boosting)   -0.0650     0.0270   
               T-learner (logistic regression)            -0.0230     0.0720   
Women's e-mail T-learner (gradient boosting)               0.0620     0.2300   
               S-learner (gradient boosting)               0.0860     0.2680   
               Class transformation (gradient boosting)    0.0600     0.2430   
               T-learner (logistic regression)             0.0870     0.2740   

                                                         capture_30  
               name                                                  
Men's e-mail   T-learner (gradient boosting)                 0.3180  
               S-learner (gradient boosting)                 0.3180  
               Class transformation (gradient boosting)      0.2890  
               T-learner (logistic regression)               0.3560  
Women's e-mail T-learner (gradient boosting)                 0.4620  
               S-learner (gradient boosting)                 0.5550  
               Class transformation (gradient boosting)      0.4780  
               T-learner (logistic regression)               0.5040

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, arm in zip(axes, config.TREATMENTS):
    res = results[arm]
    frac = np.r_[0, res["curve"]["fraction"]] * 100
    curve = np.r_[0, res["curve"][res["selected"]]] * 100
    ax.plot(frac, frac / 100 * curve[-1], color=BASELINE, ls="--", label="Random targeting")
    ax.plot(frac, curve, color=ARM_COLORS[arm], lw=2.2, label="Selected model")
    ax.set_title(ARM_NAMES[arm])
    ax.set_xlabel("Share of customers e-mailed (%)")
    ax.grid(axis="y", visible=True)
    ax.legend(loc="lower right")
axes[0].set_ylabel("Extra visits per 100 customers")
plt.show()

In [12]:
pd.concat({ARM_NAMES[arm]: res["deciles"].set_index("group")[["predicted", "observed", "ci_low", "ci_high"]] for arm, res in results.items()}, axis=1).round(3)

Men's e-mail                          Women's e-mail                   \
         predicted observed  ci_low ci_high      predicted observed  ci_low   
group                                                                         
1           0.1280   0.1520  0.1050  0.1980         0.0900   0.0530  0.0130   
2           0.0890   0.0620  0.0210  0.1040         0.0820   0.0880  0.0490   
3           0.0810   0.0530  0.0130  0.0930         0.0770   0.0870  0.0500   
4           0.0770   0.0760  0.0390  0.1120         0.0720   0.0810  0.0430   
5           0.0730   0.0320 -0.0020  0.0670         0.0660   0.0620  0.0250   
6           0.0690   0.0800  0.0450  0.1150         0.0380   0.0530  0.0180   
7           0.0650   0.0600  0.0230  0.0960         0.0230  -0.0120 -0.0380   
8           0.0620   0.0840  0.0480  0.1210         0.0160   0.0150 -0.0180   
9           0.0590   0.0750  0.0380  0.1120         0.0050   0.0210 -0.0180   
10          0.0550   0.0870  0.0520  0.1230        -0.0200   0.0090 -0.0300   

               
      ci_high  
group          
1      0.0930  
2      0.1270  
3      0.1240  
4      0.1180  
5      0.0980  
6      0.0890  
7      0.0140  
8      0.0480  
9      0.0600  
10     0.0480

In [13]:
profile = pd.DataFrame(results["womens"]["profile"]).set_index("trait")
profile.columns = ["Top 30% by predicted uplift", "Bottom 30%", "All hold-out customers"]
profile.round(3)

,Top 30% by predicted uplift,Bottom 30%,All hold-out customers
trait,,,
Bought women's items,1.0000,0.0150,0.5490
Bought men's items,0.1810,1.0000,0.5470
New customer,0.5700,0.2520,0.5020
Shops on the web only,0.6080,0.4420,0.4420
Past-year spend ($),274.3220,241.3050,242.3210
Months since last purchase,5.1080,5.2530,5.7100


**Takeaways.** For the women's e-mail every learner beats random targeting with a bootstrap interval clear of zero, and the simplest one,
a logistic-regression T-learner, generalises best. Its top 30% captures about half of the e-mail's extra visits, and the profile shows why:
the model puts women's-merchandise buyers first. For the men's e-mail no learner reliably beats random targeting, because the e-mail lifts
nearly everyone by a similar amount. There is no persuadable subgroup to find.

## 7. Which targeting policy should the business run?

Per-arm response models choose an e-mail (or none) for each customer. Because each customer was randomised with known probability (about 1/3 per arm),
**inverse-propensity weighting** estimates what any policy would have produced on the hold-out customers, without running a new experiment.
Intervals for the difference from sending the men's e-mail to everyone come from a paired bootstrap.

In [14]:
policies, info = targeting.compare_policies(train, test)
cols = ["policy", "emails_per_1000", "visit_per_1000", "visit_vs_baseline", "visit_vs_low", "visit_vs_high", "spend_per_1000"]
print(f"Share of customers for whom the women's e-mail is predicted to be better: {info['womens_share_in_best']:.0%}")
policies[cols].round(1)

Share of customers for whom the women's e-mail is predicted to be better: 29%


,policy,emails_per_1000,visit_per_1000,visit_vs_baseline,visit_vs_low,visit_vs_high,spend_per_1000
0,No e-mail,0.0000,106.2000,-76.5000,-90.3000,-63.0000,634.3000
1,Women's e-mail to everyone,"1,000.0000",151.3000,-31.4000,-44.7000,-18.3000,983.4000
2,Men's e-mail to everyone,"1,000.0000",182.7000,0.0000,0.0000,0.0000,"1,528.8000"
3,Best e-mail for each customer,"1,000.0000",178.0000,-4.7000,-12.2000,2.3000,"1,510.1000"
4,"Best e-mail, top half by predicted uplift",500.0000,150.4000,-32.4000,-43.2000,-22.0000,"1,066.2000"


Choosing the better e-mail for each customer does not beat sending the men's e-mail to everyone: the difference is −4.7 visits per 1,000
with an interval that spans zero. That matches section 5, where the women's e-mail was never clearly better for any segment. Targeting
only the top half keeps most of the extra visits with half the e-mails, which only pays if sends are expensive. The dashboard's simulator
finds the best share for any cost per e-mail and value per visit.

## 8. Power for the next test

In [15]:
n = int((df["arm"] == "control").sum())
for metric in ["visit", "conversion"]:
    p = df.loc[df["arm"] == "control", metric].mean()
    mde = experiment.mde_proportion(p, n)
    print(f"{config.METRICS[metric]:<16} baseline {p:.3%}: minimum detectable effect at {n:,} per arm = {mde * 100:+.2f} pp")
p = df.loc[df["arm"] == "control", "conversion"].mean()
print(f"Customers per arm to detect +0.25 pp in conversion: {experiment.sample_size_proportion(p, 0.0025):,}")

Visit rate       baseline 10.617%: minimum detectable effect at 21,306 per arm = +0.84 pp
Conversion rate  baseline 0.573%: minimum detectable effect at 21,306 per arm = +0.20 pp
Customers per arm to detect +0.25 pp in conversion: 17,396


## Conclusions

* **Send the men's e-mail to the whole list.** It lifts revenue by $0.77 per customer over two weeks (95% CI $0.49–$1.05), beats the
  women's e-mail on every outcome, and no per-customer choice of e-mail improves on it.
* **If sends must be cut**, rank customers with the uplift model: for the women's e-mail, the top 30% delivers about half of the extra visits.
* **The experiment was sound**: no sample-ratio mismatch, balanced covariates, and conclusions that survive multiple-testing correction,
  bootstrap intervals and regression adjustment.
* **Next test**: measure longer-term outcomes (repeat purchases, unsubscribes). A two-week window cannot show whether e-mailing
  everyone has a fatigue cost, which is the main argument for targeting.